In [1]:
import anndata
import pandas as pd
import numpy as np
import matplotlib as plt
import scvelo as scv
import scanpy
import igraph
import scanpy as sc

In [2]:
Sample_3_loom = anndata.read_loom("/data/lyh/Project/UC_CRC/Velocyto/Sample_3/velocyto/Sample_3.loom")

In [ ]:
Sample_3_loom.obs_names

In [4]:
#2
#load meta-data
sample_obs = pd.read_csv("/data/lyh/Project/UC_CRC/Velocyto/scvelo_input/Sample_3_cellID_obs.csv")
umap_cord = pd.read_csv("/data/lyh/Project/UC_CRC/Velocyto/scvelo_input/Sample_3_cell_embeddings.csv")
cell_clusters = pd.read_csv("/data/lyh/Project/UC_CRC/Velocyto/scvelo_input/Sample_3_clusters.csv")

In [5]:
Sample_3_loom.var_names_make_unique()

In [6]:
cellID_obs_Sample_3 = sample_obs[sample_obs["x"].str.contains("Sample_3")]

In [7]:
Sample_3_loom = Sample_3_loom[np.isin(Sample_3_loom.obs.index, cellID_obs_Sample_3)]

In [ ]:
total_samples_index = pd.DataFrame(Sample_3_loom.obs.index)
total_samples_index = total_samples_index.rename(columns = {'CellID':'Cell ID'})
cell_clusters = cell_clusters.rename(columns = {'CellID':'Cell ID'})
cell_clusters

In [9]:
del cell_clusters['Unnamed: 0']
cell_clusters_ordered = total_samples_index.merge(cell_clusters, on = "Cell ID")
cell_clusters_ordered = cell_clusters_ordered.iloc[:,1]

In [10]:
Sample_3_loom.obs['cell_clusters']=cell_clusters_ordered.values
umap = umap_cord.rename(columns = {'Unnamed: 0':'Cell ID'})
umap_ordered = total_samples_index.merge(umap, on = "Cell ID")
total_samples_index
umap_ordered = umap_ordered.iloc[:,1:]
Sample_3_loom.obsm['X_umap'] = umap_ordered.values
Sample_3_loom.obsm['X_umap'] = Sample_3_loom.obsm['X_umap'].astype('float32')

In [ ]:
scv.pp.filter_and_normalize(Sample_3_loom)
scv.pp.moments(Sample_3_loom)
scv.tl.recover_dynamics(Sample_3_loom,n_jobs = 10)
scv.tl.velocity_graph(Sample_3_loom)
scv.tl.velocity(Sample_3_loom, mode='dynamical')
scv.tl.velocity_graph(Sample_3_loom)
scv.tl.latent_time(Sample_3_loom)
scv.tl.velocity_confidence(Sample_3_loom)

In [ ]:
scv.pl.velocity_embedding_stream(Sample_3_loom, basis='umap',color='cell_clusters',
                                 palette = ["#FF0000","#FFA501"],
                                size = 200,
                                dpi = 300,legend_loc='none', alpha = 1,
                                figsize = (3,4))